# Predicción de Informalidad Laboral — Visualización
**Datos:** GEIH 2024  
**Objetivo:** Generar visualizaciones interactivas (Plotly) que consume la app Streamlit (Fase 5)  
**Requiere:** Ejecutar primero `4.Modelamiento.ipynb` para tener `outputs/` disponible

---
### Setup y carga de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import pickle
import shap
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_curve, auc

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

OUT_DIR     = Path('outputs')
PARQUET_DIR = Path('parquet')

# Dataset crudo validado (generado por Fase 4)
df = pd.read_parquet(OUT_DIR / 'datos_procesados.parquet')

# Modelo y metadatos
with open(OUT_DIR / 'champion_geih.pkl', 'rb') as f:
    modelo = pickle.load(f)
with open(OUT_DIR / 'champion_geih_meta.json') as f:
    meta = json.load(f)

# Test set para curva ROC
df_test = pd.read_parquet(PARQUET_DIR / 'test.parquet')
X_test  = df_test.drop('INFORMAL', axis=1)
y_test  = df_test['INFORMAL'].astype(int)

print(f'Dataset: {len(df):,} registros')
print(f'Modelo campeón: {meta["model"]}  |  F1: {meta["test_f1"]:.4f}  |  AUC: {meta["test_auc"]:.4f}')

c:\Users\Juli\Desktop\Proyecto_Final_maestria\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: 351,721 registros
Modelo campeón: LightGBM (tuned)  |  F1: 0.9593  |  AUC: 0.9506


---
### Distribución de la variable objetivo

In [2]:
n_total    = len(df)
n_informal = (df['INFORMAL'] == 1).sum()
n_formal   = (df['INFORMAL'] == 0).sum()
tasa_pond  = (df['INFORMAL'] * df['FEX_C18']).sum() / df['FEX_C18'].sum() * 100

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Conteo (muestra)', 'Proporción ponderada (FEX_C18)'],
                    specs=[[{'type':'bar'}, {'type':'pie'}]])

fig.add_trace(go.Bar(
    x=['Formal (0)', 'Informal (1)'],
    y=[n_formal, n_informal],
    marker_color=['#2E75B6', '#E05C5C'],
    text=[f'{n_formal:,}', f'{n_informal:,}'],
    textposition='outside'
), row=1, col=1)

w_f = df[df['INFORMAL']==0]['FEX_C18'].sum()
w_i = df[df['INFORMAL']==1]['FEX_C18'].sum()
fig.add_trace(go.Pie(
    labels=['Formal', 'Informal'],
    values=[w_f, w_i],
    marker_colors=['#2E75B6', '#E05C5C']
), row=1, col=2)

fig.update_layout(
    title=f'Distribución INFORMAL — GEIH 2024 · Tasa ponderada: {tasa_pond:.1f}%',
    showlegend=False, height=380
)
fig.show()
fig.write_html(OUT_DIR / 'viz_01_target.html')
print(f'Informalidad ponderada (DANE): {tasa_pond:.1f}%')

Informalidad ponderada (DANE): 56.0%


---
### Informalidad por departamento

In [3]:
NOMBRES_DPTO = {
     5:'Antioquia',     8:'Atlántico',     11:'Bogotá D.C.',
    13:'Bolívar',       15:'Boyacá',        17:'Caldas',
    18:'Caquetá',       19:'Cauca',         20:'Cesar',
    23:'Córdoba',       25:'Cundinamarca',  27:'Chocó',
    41:'Huila',         44:'La Guajira',    47:'Magdalena',
    50:'Meta',          52:'Nariño',        54:'N. Santander',
    63:'Quindío',       66:'Risaralda',     68:'Santander',
    70:'Sucre',         73:'Tolima',        76:'Valle del Cauca',
    81:'Arauca',        85:'Casanare',      86:'Putumayo',
    88:'San Andrés',    91:'Amazonas',      94:'Guainía',
    95:'Guaviare',      97:'Vaupés',        99:'Vichada',
}

def tasa_pond(g):
    return (g['INFORMAL'] * g['FEX_C18']).sum() / g['FEX_C18'].sum() * 100

tasa_dpto = (
    df[df['DPTO'].notna()]
    .groupby('DPTO', observed=True)
    .apply(tasa_pond, include_groups=False)
    .reset_index()
    .rename(columns={0: 'tasa'})
    .sort_values('tasa')
)
tasa_dpto['nombre'] = tasa_dpto['DPTO'].map(NOMBRES_DPTO).fillna('DPTO ' + tasa_dpto['DPTO'].astype(str))

fig = px.bar(
    tasa_dpto, x='tasa', y='nombre', orientation='h',
    title='Tasa de informalidad por departamento — GEIH 2024 (ponderada)',
    labels={'tasa': '% Informal', 'nombre': ''},
    color='tasa', color_continuous_scale='RdYlGn_r',
    text=tasa_dpto['tasa'].round(1).astype(str) + '%',
    height=750
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False)
fig.show()
fig.write_html(OUT_DIR / 'viz_02_dpto.html')

---
### Informalidad por posición ocupacional y zona

In [4]:
LABELS_POS = {
    1:'Empleado particular', 2:'Empleado gobierno', 3:'Empleado doméstico',
    4:'Cuenta propia', 5:'Empleador', 6:'Familiar sin remuneración',
    7:'Jornalero/Peón', 8:'Otro'
}

tasa_pos = (
    df[df['P6430'].notna()]
    .groupby('P6430', observed=True)
    .apply(tasa_pond, include_groups=False)
    .reset_index().rename(columns={0:'tasa'})
    .sort_values('tasa')
)
tasa_pos['etiqueta'] = tasa_pos['P6430'].map(LABELS_POS)

t_zona = (
    df[df['CLASE'].notna()]
    .groupby('CLASE', observed=True)
    .apply(tasa_pond, include_groups=False)
    .reset_index().rename(columns={0:'tasa'})
)
t_zona['etiqueta'] = t_zona['CLASE'].map({1:'Cabecera municipal', 2:'Rural'})

fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35],
                    subplot_titles=['Por posición ocupacional', 'Por zona'])

colors_pos = [f'rgb({int(255*(v/100))},{int(200*(1-v/100))},60)' for v in tasa_pos['tasa']]
fig.add_trace(go.Bar(
    x=tasa_pos['tasa'], y=tasa_pos['etiqueta'],
    orientation='h', marker_color=colors_pos,
    text=tasa_pos['tasa'].round(1).astype(str)+'%', textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=t_zona['etiqueta'], y=t_zona['tasa'],
    marker_color=['#5DCAA5','#EF9F27'],
    text=t_zona['tasa'].round(1).astype(str)+'%', textposition='outside'
), row=1, col=2)

fig.update_layout(title='Informalidad por posición ocupacional y zona (ponderada)',
                  showlegend=False, height=420)
fig.show()
fig.write_html(OUT_DIR / 'viz_03_posicion_zona.html')

---
### Informalidad por educación y edad

In [5]:
LABELS_EDU = {
    1:'Ninguno', 2:'Preescolar', 3:'Primaria inc.', 4:'Primaria',
    5:'Sec. inc.', 6:'Secundaria', 7:'Media inc.', 8:'Media',
    9:'Técnica/Tecn.', 10:'Universitaria', 11:'Especialización',
    12:'Maestría', 13:'Doctorado'
}

tasa_edu = (
    df[df['P3042'].notna()]
    .groupby('P3042', observed=True)
    .apply(tasa_pond, include_groups=False)
    .reset_index().rename(columns={0:'tasa'})
)
tasa_edu['etiqueta'] = tasa_edu['P3042'].map(LABELS_EDU)

# Quintetos de edad
df_age = df[df['P6040'].between(15, 74)].copy()
df_age['EDAD_Q'] = pd.cut(df_age['P6040'], bins=range(14,76,5),
                           labels=[f'{i}-{i+4}' for i in range(15,75,5)])
tasa_edad = (
    df_age[df_age['EDAD_Q'].notna()]
    .groupby('EDAD_Q', observed=True)
    .apply(tasa_pond, include_groups=False)
    .reset_index().rename(columns={0:'tasa'})
)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Por nivel educativo', 'Por grupo de edad'])

fig.add_trace(go.Bar(
    x=tasa_edu['etiqueta'], y=tasa_edu['tasa'],
    marker_color=tasa_edu['tasa'],
    marker_colorscale='RdYlGn_r',
    text=tasa_edu['tasa'].round(0).astype(int).astype(str)+'%',
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=tasa_edad['EDAD_Q'].astype(str), y=tasa_edad['tasa'],
    mode='lines+markers', line=dict(color='#E05C5C', width=2),
    marker=dict(size=7),
    fill='tozeroy', fillcolor='rgba(224,92,92,0.12)'
), row=1, col=2)

fig.update_layout(title='Informalidad por educación y edad (ponderada)',
                  showlegend=False, height=400)
fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.show()
fig.write_html(OUT_DIR / 'viz_04_edu_edad.html')

---
### Métricas del modelo — Curva ROC y comparativa

In [6]:
# Curva ROC interactiva
y_prob = modelo.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f'ROC (AUC = {roc_auc_val:.4f})',
    line=dict(color='#2E75B6', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash'), name='Baseline'
))
fig_roc.update_layout(
    title=f'Curva ROC — {meta["model"]} (AUC = {roc_auc_val:.4f})',
    xaxis_title='Tasa de falsos positivos',
    yaxis_title='Tasa de verdaderos positivos',
    height=450, width=600
)
fig_roc.show()
fig_roc.write_html(OUT_DIR / 'viz_05_roc.html')

# Tabla de métricas
try:
    import pandas as pd
    comp = pd.read_csv(OUT_DIR / 'metrics_comparison.csv')
    print('Comparativa de modelos:')
    print(comp.to_string(index=False, float_format='{:.4f}'.format))
except:
    print('metrics_comparison.csv no encontrado — ejecuta 4.Modelamiento.ipynb')

Comparativa de modelos:
             Modelo     F1  AUC-ROC  Precision  Recall
   LightGBM (tuned) 0.9593   0.9506     0.9526  0.9661
            XGBoost 0.9525   0.9475     0.9585  0.9465
      Random Forest 0.9458   0.9461     0.9631  0.9291
Logistic Regression 0.9283   0.9293     0.9632  0.8958


---
### SHAP — Importancia de variables (Plotly interactivo)

In [7]:
SAMPLE_SHAP = 2000
X_shap = X_test.sample(SAMPLE_SHAP, random_state=42)

explainer   = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_shap)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Nombres legibles — cubre todas las columnas OHE del preprocessor
NOMBRES = {
    # Numéricas
    'P6040':     'Edad',
    'ANIOS_EDU': 'Años de educación',
    'P6800':     'Horas / semana',
    # Binarias
    'P3271':          'Sexo',
    'CLASE':          'Zona (urbana/rural)',
    'CUENTA_PROPIA':  'Cuenta propia',
    'MICROEMPRESA':   'Microempresa (≤10 p.)',
    'SUBEMPLEADO':    'Subempleado (<32 h)',
    'PLURIEMPLEO':    'Pluriempleo',
    'CONTRATO_VERBAL':'Contrato verbal',
    # Target encoding
    '0': 'Rama de actividad',
    '1': 'Departamento',
    # OHE — Grupo de edad
    'EDAD_GRUPO_15-24': 'Edad: 15–24',
    'EDAD_GRUPO_25-34': 'Edad: 25–34',
    'EDAD_GRUPO_35-44': 'Edad: 35–44',
    'EDAD_GRUPO_45-54': 'Edad: 45–54',
    'EDAD_GRUPO_55-64': 'Edad: 55–64',
    'EDAD_GRUPO_65+':   'Edad: 65+',
    # OHE — Estado civil (P6070)
    'P6070_1.0': 'Civil: No unido/a',
    'P6070_2.0': 'Civil: Unión libre',
    'P6070_3.0': 'Civil: Casado/a',
    'P6070_4.0': 'Civil: Separado/a',
    'P6070_5.0': 'Civil: Viudo/a',
    'P6070_6.0': 'Civil: NS/NR',
    # OHE — Posición ocupacional (P6430) — modelo anterior
    'P6430_1': 'Posición: Empleado particular',
    'P6430_2': 'Posición: Empleado gobierno',
    'P6430_3': 'Posición: Doméstico',
    'P6430_4': 'Posición: Cuenta propia',
    'P6430_5': 'Posición: Empleador',
    'P6430_6': 'Posición: Familiar s/rem.',
    'P6430_7': 'Posición: Jornalero',
    'P6430_8': 'Posición: Otro',
    # OHE — Tipo de contrato (P6450)
    'P6450_1.0': 'Contrato: Verbal',
    'P6450_2.0': 'Contrato: Escrito',
    'P6450_9.0': 'Contrato: NS/NR',
    # OHE — Tamaño del establecimiento (P3069)
    'P3069_1':  'Establ. 1 persona',
    'P3069_2':  'Establ. 2–5 personas',
    'P3069_3':  'Establ. 6–10 personas',
    'P3069_4':  'Establ. 11–19 personas',
    'P3069_5':  'Establ. 20–30 personas',
    'P3069_6':  'Establ. 31–50 personas',
    'P3069_7':  'Establ. 51–100 personas',
    'P3069_8':  'Establ. 101–200 personas',
    'P3069_9':  'Establ. 201+ personas',
    'P3069_10': 'Establ. Tamaño NS',
}
feat_names = [NOMBRES.get(c, c) for c in X_shap.columns]

# Importancia media |SHAP| como barras Plotly
mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feat_names).sort_values(ascending=True)
mean_abs = mean_abs.tail(20)  # top 20

fig_shap = px.bar(
    x=mean_abs.values, y=mean_abs.index, orientation='h',
    title=f'Importancia de variables (|SHAP| medio) — {meta["model"]}',
    labels={'x': 'Importancia media |SHAP|', 'y': ''},
    color=mean_abs.values, color_continuous_scale='Blues',
    height=600,
)
fig_shap.update_layout(
    coloraxis_showscale=False,
    yaxis=dict(tickfont=dict(size=13, family='Arial')),
    xaxis=dict(tickfont=dict(size=12)),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=210, r=30, t=50, b=40),
)
fig_shap.show()
fig_shap.write_html(OUT_DIR / 'viz_06_shap.html')

print('Top 10 features por importancia SHAP:')
print(mean_abs.tail(10).sort_values(ascending=False).to_string())

Top 10 features por importancia SHAP:
Años de educación        1.315443
Rama de actividad        0.912441
Departamento             0.863869
Edad                     0.716386
Microempresa (≤10 p.)    0.637492
Horas / semana           0.443175
Establ. Tamaño NS        0.286056
Establ. 1 persona        0.279147
Sexo                     0.263248
Zona (urbana/rural)      0.161685


---
### Resumen de outputs generados

In [8]:
print('Archivos HTML generados en outputs/:')
for f in sorted(OUT_DIR.glob('viz_*.html')):
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<35} {kb:.0f} KB')

print()
print('Próximo paso → ejecutar app.py con Streamlit:')
print('  streamlit run app.py')

Archivos HTML generados en outputs/:
  viz_01_target.html                  4741 KB
  viz_02_dpto.html                    4742 KB
  viz_03_posicion_zona.html           4741 KB
  viz_04_edu_edad.html                4742 KB
  viz_05_roc.html                     4830 KB
  viz_06_shap.html                    4742 KB

Próximo paso → ejecutar app.py con Streamlit:
  streamlit run app.py
